In [1]:
!pip uninstall tensorflow -y

Found existing installation: tensorflow 2.19.0
Uninstalling tensorflow-2.19.0:
  Successfully uninstalled tensorflow-2.19.0


In [2]:
!pip install --no-deps --no-index /kaggle/input/wheelf/sacremoses-0.1.1-py3-none-any.whl

Processing /kaggle/input/wheelf/sacremoses-0.1.1-py3-none-any.whl


In [3]:
import pandas as pd
import torch
from transformers import MarianMTModel, MarianTokenizer
from tqdm import tqdm
import os
import re
import sys

In [4]:
MODEL_PATH = "/kaggle/input/akkadian-model/pytorch/default/1"
TEST_FILE = "/kaggle/input/deep-past-initiative-machine-translation/test.csv"
PN_FILE = "/kaggle/input/akkadian-translation/proper_nouns.csv"
SUBMISSION_FILE = "submission.csv"

In [5]:
def clean_akkadian(text):
    if not isinstance(text, str): return ""
    # Normalize Gaps
    text = re.sub(r'\[x\]', ' <gap> ', text)
    text = re.sub(r'…|\[…\s?…\]|\[\.\.\.\]', ' <big_gap> ', text)
    # Remove uncertain markers (! ? / ˹ ˺)
    text = re.sub(r'[!?:/˹˺]', '', text)
    # Clean Brackets: [text] -> text
    text = re.sub(r'\[([^\]]*)\]', r'\1', text)
    # Normalize Determinatives: (d) -> {d}, (ki) -> {ki}
    text = re.sub(r'\(([a-zA-Z0-9]+)\)', r'{\1}', text)
    
    # Normalize Subscripts: il5 -> il₅
    subscripts = str.maketrans("0123456789", "₀₁₂₃₄₅₆₇₈₉")
    def sub_repl(m): return m.group(1) + m.group(2).translate(subscripts)
    text = re.sub(r'([a-zA-Z])([0-9])', sub_repl, text)
    
    # Collapse spaces
    return re.sub(r'\s+', ' ', text).strip()

In [6]:
def translate_with_optimal_beams(texts, model, tokenizer, device):
    """Test different beam configurations"""
    
    configs = [
        {'beams': 8, 'length_penalty': 0.8, 'no_repeat_ngram': 3},
        {'beams': 10, 'length_penalty': 1.0, 'no_repeat_ngram': 3},
        {'beams': 12, 'length_penalty': 1.2, 'no_repeat_ngram': 2},
    ]
    
    all_translations = []
    
    for config in configs:
        batch_translations = []
        for i in range(0, len(texts), 32):
            batch = texts[i:i+32]
            encoded = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
            
            with torch.no_grad():
                generated = model.generate(
                    **encoded,
                    max_length=128,
                    num_beams=config['beams'],
                    length_penalty=config['length_penalty'],
                    no_repeat_ngram_size=config['no_repeat_ngram'],
                    early_stopping=True
                )
            
            decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
            batch_translations.extend(decoded)
        
        all_translations.append(batch_translations)
    
    # Vote or pick best
    return all_translations[1]


In [7]:
def enhanced_postprocess(translation, source_text, proper_nouns):
    """Improved post-processing"""
    
    # 1. Capitalize first letter
    if translation and len(translation) > 0:
        translation = translation[0].upper() + translation[1:]
    
    # 2. Fix proper nouns from source
    source_caps = re.findall(r'\b[A-Z][a-z]+\b', source_text)
    for proper in source_caps:
        # Case-insensitive replace
        pattern = re.compile(re.escape(proper), re.IGNORECASE)
        translation = pattern.sub(proper, translation)
    
    # 3. Apply known proper nouns
    words = translation.split()
    processed_words = []
    for i, w in enumerate(words):
        clean_w = w.strip(".,!?\"'").title()
        if clean_w in proper_nouns or (i == 0 and len(w) > 0):
            # Keep capitalization for proper nouns and first word
            if i == 0:
                processed_words.append(w[0].upper() + w[1:] if len(w) > 1 else w.upper())
            else:
                processed_words.append(w.title())
        else:
            processed_words.append(w)
    translation = " ".join(processed_words)
    
    # 4. Clean spacing
    translation = re.sub(r'\s+', ' ', translation).strip()
    translation = re.sub(r'\s+([.,!?])', r'\1', translation)  # Remove space before punctuation
    translation = re.sub(r'([.,!?])([A-Za-z])', r'\1 \2', translation)  # Add space after punctuation
    
    # 5. Ensure ends with punctuation
    if translation and translation[-1] not in '.!?':
        translation += '.'
    
    return translation

In [8]:
def generate_submission():
    # 1. Load Resources
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Loading model on {device}...")
    
    if not os.path.exists(MODEL_PATH):
        print(f"CRITICAL ERROR: Model path {MODEL_PATH} not found.")
        pd.DataFrame({'id': [], 'translation': []}).to_csv(SUBMISSION_FILE, index=False)
        return

    try:
        model = MarianMTModel.from_pretrained(MODEL_PATH).to(device)
        tokenizer = MarianTokenizer.from_pretrained(MODEL_PATH)
        model.eval()  # Set to eval mode
    except Exception as e:
        print(f"Error loading model: {e}")
        return

    # 2. Load test data
    print("Loading and cleaning test data...")
    try:
        test_df = pd.read_csv(TEST_FILE)
    except FileNotFoundError:
        test_df = pd.read_csv("/kaggle/input/deep-past-initiative-machine-translation/test.csv")

    test_df['clean_transliteration'] = test_df['transliteration'].apply(clean_akkadian)
    
    # 3. Load proper nouns
    proper_nouns = set()
    if os.path.exists(PN_FILE):
        pn_df = pd.read_csv(PN_FILE)
        proper_nouns = set(pn_df['norm'].dropna().apply(lambda x: str(x).title()).tolist())
        print(f"Loaded {len(proper_nouns)} proper nouns")

    # 4. Generate translations with optimal beams
    # **THIS IS THE KEY CHANGE - REPLACE OLD LOOP**
    print(f"Generating translations for {len(test_df)} items...")
    inputs = test_df['clean_transliteration'].tolist()
    
    # Use the new function instead of the old loop
    translations = translate_with_optimal_beams(inputs, model, tokenizer, device)

    # 5. Enhanced post-processing
    print("Applying enhanced post-processing...")
    final_output = []
    for i, text in enumerate(translations):
        source = test_df.iloc[i]['clean_transliteration']
        processed = enhanced_postprocess(text, source, proper_nouns)
        final_output.append(processed)

    # 6. Save submission
    if len(final_output) != len(test_df):
        print(f"WARNING: Mismatch in lengths. Input: {len(test_df)}, Output: {len(final_output)}")
        while len(final_output) < len(test_df):
            final_output.append("")
            
    submission = pd.DataFrame({'id': test_df['id'], 'translation': final_output})
    submission.to_csv(SUBMISSION_FILE, index=False)
    print(f"✓ Submission saved to {SUBMISSION_FILE} with {len(submission)} rows.")
    print(f"✓ Used optimized beam search with 3 configurations")

In [9]:
generate_submission()

Loading model on cuda...
Loading and cleaning test data...
Loaded 6384 proper nouns
Generating translations for 4 items...
Applying enhanced post-processing...
✓ Submission saved to submission.csv with 4 rows.
✓ Used optimized beam search with 3 configurations


In [10]:
check_csv = pd.read_csv('/kaggle/working/submission.csv')
check_csv.head()

,id,translation
0,0,"Thus Kaniya, say to A-qēl, the datum of our fa..."
1,1,The Kanesh colony gave us for these proceeding...
2,2,"If you have written me there, either to the Ci..."
3,3,The very day you hear our letter Aššur-taklāku...
